In [1]:
import numpy as np
import scipy.sparse as sp
import tensorflow as tf
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout,Input,Reshape,BatchNormalization
from tensorflow.keras.losses import CategoricalCrossentropy, BinaryCrossentropy
from tensorflow.keras.metrics import categorical_accuracy, binary_accuracy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam,SGD
from tensorflow.keras.regularizers import L1, L2, l2
from tensorflow.keras import layers

import spektral
from spektral.data import MixedLoader
from spektral.data import Dataset, DisjointLoader, Graph
from spektral.layers import GCSConv, GlobalAvgPool, GATConv, DiffPool, GlobalAttentionPool
from spektral.layers.pooling import TopKPool
from spektral.transforms.normalize_adj import NormalizeAdj
from spektral.utils.sparse import sp_matrix_to_sp_tensor
import tensorflow.keras.backend as K
import tensorflow_addons as tfa


C:\Users\yjm85\Anaconda3\envs\Immunotherapy4\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:67: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.9.0 and strictly below 2.12.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.5.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/addons
  UserWarning,


In [2]:
class MyDataset(Dataset):
    """
    """
    def __init__(self, datafromnpzfile, **kwargs):
        self.a=None
        self.file=datafromnpzfile
        self.x=datafromnpzfile['x']
        self.y = datafromnpzfile['y']
        self.info = datafromnpzfile['info']
        self.cols=datafromnpzfile['cols']
        self.n=self.x.shape[0]
        super().__init__(**kwargs)
    def read(self):
        self.a=sp.csr_matrix(self.file['pathway_a'])
        graph=[]
        for i in range(self.n):
            x=self.x[i]
            y=self.y[i]
            graph.append(Graph(x=x,y=y))
        # We must return a list of Graph objects
        return graph

In [3]:
import os
import re
import pandas as pd


# data_dir = os.path.dirname(__file__)
class GMT():
    # genes_cols : start reading genes from genes_col(default 1, it can be 2 e.g. if an information col is added after the pathway col)
    # pathway col is considered to be the first column (0)
    def load_data(self, filename, genes_col=1, pathway_col=0):

        data_dict_list = []
        with open(filename) as gmt:

            data_list = gmt.readlines()

            # print data_list[0]
            for row in data_list:
                genes = row.strip().split('\t')
                genes = [re.sub('_copy.*', '', g) for g in genes]
                genes = [re.sub('\\n.*', '', g) for g in genes]
                for gene in genes[genes_col:]:
                    pathway = genes[pathway_col]
                    dict = {'group': pathway, 'gene': gene}
                    data_dict_list.append(dict)

        df = pd.DataFrame(data_dict_list)
        # print df.head()

        return df

    def load_data_dict(self, filename):

        data_dict_list = []
        dict = {}
        with open(os.path.join(data_dir, filename)) as gmt:
            data_list = gmt.readlines()

            # print data_list[0]
            for row in data_list:
                genes = row.split('\t')
                dict[genes[0]] = genes[2:]

        return dict

    def write_dict_to_file(self, dict, filename):
        lines = []
        with open(filename, 'w') as gmt:
            for k in dict:
                str1 = '	'.join(str(e) for e in dict[k])
                line = str(k) + '	' + str1 + '\n'
                lines.append(line)
            gmt.writelines(lines)
        return

    def __init__(self):

        return


In [4]:
import logging
import numpy as np
import pandas as pd

def get_KEGG_map(input_list, filename='c2.cp.kegg.v6.1.symbols.gmt', genes_col=1, shuffle_genes=False):
    '''
    :param input_list: list of inputs under consideration (e.g. genes)
    :param filename: a gmt formated file e.g. pathway1 gene1 gene2 gene3
#                                     pathway2 gene4 gene5 gene6
    :param genes_col: the start index of the gene columns
    :param shuffle_genes: {True, False}
    :return: dataframe with rows =genes and columns = pathways values = 1 or 0 based on the membership of certain gene in the corresponding pathway
    '''
    d = GMT()
    df = d.load_data(filename, genes_col)
    df['value'] = 1
    mapp = pd.pivot_table(df, values='value', index='gene', columns='group', aggfunc=np.sum)
    mapp = mapp.fillna(0)
    cols_df = pd.DataFrame(index=input_list)
    mapp = cols_df.merge(mapp, right_index=True, left_index=True, how='left')
    mapp = mapp.fillna(0)
    genes = mapp.index
    pathways = mapp.columns
    mapp = mapp.values

    if shuffle_genes:
        logging.info('shuffling')
        ones_ratio = np.sum(mapp) / np.prod(mapp.shape)
        logging.info('ones_ratio {}'.format(ones_ratio))
        mapp = np.random.choice([0, 1], size=mapp.shape, p=[1 - ones_ratio, ones_ratio])
        logging.info('random map ones_ratio {}'.format(ones_ratio))
    return mapp, genes, pathways


In [5]:
import numpy as np
from tensorflow.keras.layers import Layer
from tensorflow.keras import regularizers
from tensorflow.keras.initializers import glorot_uniform, Initializer
from tensorflow.keras import activations
from tensorflow.keras import initializers, constraints
from tensorflow.keras.regularizers import Regularizer

class SparseTF(Layer):
    def __init__(self, units, map=None, nonzero_ind=None, kernel_initializer='glorot_uniform', W_regularizer=None,
                 activation='tanh', use_bias=True,
                 bias_initializer='zeros', bias_regularizer=None, kernel_constraint=None, bias_constraint=None,
                 **kwargs):
        self.units = units
        self.activation = activation
        self.map = map
        self.nonzero_ind = nonzero_ind
        self.use_bias = use_bias
        self.kernel_initializer = initializers.get(kernel_initializer)
        self.kernel_regularizer = regularizers.get(W_regularizer)
        self.bias_initializer = initializers.get(bias_initializer)
        self.bias_regularizer = regularizers.get(bias_regularizer)
        self.activation_fn = activations.get(activation)
        self.kernel_constraint = constraints.get(kernel_constraint)
        self.bias_constraint = constraints.get(bias_constraint)
        super(SparseTF, self).__init__(**kwargs)

    def build(self, input_shape):
        input_dim = input_shape[1]
        # random sparse constarints on the weights
        # if self.map is None:
        #     mapp = np.random.rand(input_dim, self.units)
        #     mapp = mapp > 0.9
        #     mapp = mapp.astype(np.float32)
        #     self.map = mapp
        # else:
        if not self.map is None:
            self.map = self.map.astype(np.float32)

        # can be initialized directly from (map) or using a loaded nonzero_ind (useful for cloning models or create from config)
        if self.nonzero_ind is None:
            nonzero_ind = np.array(np.nonzero(self.map)).T
            self.nonzero_ind = nonzero_ind

        self.kernel_shape = (input_dim, self.units)
        # sA = sparse.csr_matrix(self.map)
        # self.sA=sA.astype(np.float32)
        # self.kernel_sparse = tf.SparseTensor(self.nonzero_ind, sA.data, sA.shape)

        # self.kernel_shape = (input_dim, self.units)
        # sA = sparse.csr_matrix(self.map)
        # self.sA=sA.astype(np.float32)
        # self.kernel_sparse = tf.SparseTensor(self.nonzero_ind, sA.data, sA.shape)
        # self.kernel_dense = tf.Variable(self.map)

        nonzero_count = self.nonzero_ind.shape[0]

        # initializer = initializers.get('uniform')
        # print 'nonzero_count', nonzero_count
        # self.kernel_vector = K.variable(initializer((nonzero_count,)), dtype=K.floatx(), name='kernel' )

        self.kernel_vector = self.add_weight(name='kernel_vector',
                                             shape=(nonzero_count,),
                                             initializer=self.kernel_initializer,
                                             regularizer=self.kernel_regularizer,
                                             trainable=True, constraint=self.kernel_constraint)
        # self.kernel = tf.scatter_nd(self.nonzero_ind, self.kernel_vector, self.kernel_shape, name='kernel')
        # --------
        # init = np.random.rand(input_shape[1], self.units).astype( np.float32)
        # sA = sparse.csr_matrix(init)
        # self.kernel = K.variable(sA, dtype=K.floatx(), name= 'kernel',)
        # self.kernel_vector = K.variable(init, dtype=K.floatx(), name= 'kernel',)

        # print self.kernel.values
        # ind = np.array(np.nonzero(init))
        # stf = tf.SparseTensor(ind.T, sA.data, sA.shape)
        # print stf.dtype
        # print init.shape
        # # self.kernel = stf
        # self.kernel = tf.keras.backend.variable(stf, dtype='SparseTensor', name='kernel')
        # print self.kernel.values

        if self.use_bias:
            self.bias = self.add_weight(shape=(self.units,),
                                        initializer=self.bias_initializer,
                                        name='bias',
                                        regularizer=self.bias_regularizer,
                                        constraint=self.bias_constraint)
        else:
            self.bias = None

        super(SparseTF, self).build(input_shape)  # Be sure to call this at the end
        # self.trainable_weights = [self.kernel_vector]

    def call(self, inputs):
        # print self.kernel_vector.shape, inputs.shape
        # print self.kernel_shape, self.kernel_vector
        # print self.nonzero_ind
        # kernel_sparse= tf.S parseTensor(self.nonzero_ind, self.kernel_vector, self.kernel_shape)
        # pr = cProfile.Profile()
        # pr.enable()

        # print self.kernel_vector
        # self.kernel_sparse._values = self.kernel_vector
        tt = tf.scatter_nd(self.nonzero_ind, self.kernel_vector, self.kernel_shape)
        # print tt
        # update  = self.kernel_vector
        # tt= tf.scatter_add(self.kernel_dense, self.nonzero_ind, update)
        # tt= self.kernel_dense
        # tt[self.nonzero_ind].assign( self.kernel_vector)
        # self.kernel_dense[self.nonzero_ind] = self.kernel_vector
        # tt= tf.sparse.transpose(self.kernel_sparse)
        # output = tf.sparse.matmul(tt, tf.transpose(inputs ))
        # output = tf.matmul(tt, inputs )
        output = K.dot(inputs, tt)
        # pr.disable()
        # pr.print_stats(sort="time")
        # return tf.transpose(output)
        if self.use_bias:
            output = K.bias_add(output, self.bias)
        if self.activation_fn is not None:
            output = self.activation_fn(output)

        return output

    def get_config(self):
        config = {
            'units': self.units,
            'activation': self.activation,
            # 'kernel_shape': self.kernel_shape,
            'use_bias': self.use_bias,
            'nonzero_ind': np.array(self.nonzero_ind),
            # 'kernel_initializer': initializers.serialize(self.kernel_initializer),
            # 'kernel_regularizer': regularizers.serialize(self.kernel_regularizer),
            'bias_initializer': initializers.serialize(self.bias_initializer),
            'bias_regularizer': regularizers.serialize(self.bias_regularizer),

            'kernel_initializer': initializers.serialize(self.kernel_initializer),
            'W_regularizer': regularizers.serialize(self.kernel_regularizer),

        }
        base_config = super(SparseTF, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

    # def call(self, inputs):
    #     print self.kernel.shape, inputs.shape
    #     tt= tf.sparse.transpose(self.kernel)
    #     output = tf.sparse.matmul(tt, tf.transpose(inputs ))
    #     return tf.transpose(output)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.units)

    # def get_weights(self):
    #
    #     return [self.kernel_vector, self.bias]

In [6]:
genes=[]
with open('./genelist_8080.txt','r') as reader:
    line = reader.readline().strip()
    while line != '':  # The EOF char is an empty string
        genes.append(line)
        line = reader.readline().strip()
genes=np.array(genes)

In [7]:
mapp, genes, pathways = get_KEGG_map(genes, filename='./Kegg/human_KeggPathwayGene.gmt')
n_genes, n_pathways = mapp.shape



def build_model(mapp, n_genes, n_pathways, mapping_l1_reg = 2.5e-3, gat1_l2_reg = 2.5e-3, gat2_l2_reg = 2.5e-3, pool_l1_reg = 2.5e-3, x_dropRate = 0.3, 
                gat1_dropRate = 0.3, gat2_dropRate = 0.3, gat1_channel = 3, gat1_nhead = 3, gat2_channel = 6, pool_channel = 12, dense_channel = 6):
    x_in = Input(shape=(n_genes,))
    x_drop1 = Dropout(x_dropRate)(x_in)
    mapping_layer = SparseTF(n_pathways, mapp, activation='elu', W_regularizer=L1(mapping_l1_reg),
                                name='mapping', kernel_initializer='glorot_uniform',
                                use_bias=True)
    layer2_output = mapping_layer(x_drop1)
    layer2_res=Reshape([n_pathways,1])(layer2_output)
    a_in = Input(shape=(n_pathways,),sparse=True)
    x_1 = GATConv(
        gat1_channel,
        attn_heads=gat1_nhead,
        concat_heads=False,
        activation="tanh",
        return_attn_coef=False,
        dropout_rate=gat1_dropRate,
        kernel_regularizer=l2(gat1_l2_reg),
        attn_kernel_regularizer=l2(gat1_l2_reg),
        bias_regularizer=l2(gat1_l2_reg),
        bias_initializer='glorot_uniform',
    )([layer2_res, a_in])
    x1bn = layers.BatchNormalization()(x_1)
    x_2,att = GATConv(
        gat2_channel,
        attn_heads=1,
        concat_heads=True,
        activation="tanh",
        return_attn_coef=True,
        dropout_rate=gat2_dropRate,
        kernel_regularizer=l2(gat2_l2_reg),
        attn_kernel_regularizer=l2(gat2_l2_reg),
        bias_regularizer=l2(gat2_l2_reg),
        bias_initializer='glorot_uniform',
    )([x1bn, a_in])
    x2bn = layers.BatchNormalization()(x_2)
    attpool=GlobalAttentionPool(pool_channel, kernel_initializer='glorot_uniform', bias_initializer='zeros', kernel_regularizer=L1(pool_l1_reg))(x2bn)
    x_fc1 = Dense(dense_channel, activation="elu")(attpool)
    output = Dense(2, activation="softmax")(x_fc1)  # MNIST has 10 classes
    model = Model(inputs=[x_in, a_in], outputs=output)
    #optimizer = Adam(lr=1e-1)
    return model



mapping_l1_reg = 2.5e-3
gat1_l2_reg = 2.5e-3
gat2_l2_reg = 2.5e-3
pool_l1_reg = 2.5e-3
x_dropRate = 0.3
gat1_dropRate = 0.3
gat2_dropRate = 0.3
gat1_channel = 3
gat1_nhead = 3
gat2_channel = 6
pool_channel = 12
dense_channel = 6


model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg = 2.5e-3, gat1_l2_reg = 2.5e-3, gat2_l2_reg = 2.5e-3, pool_l1_reg = 2.5e-3, x_dropRate = 0.3, 
                gat1_dropRate = 0.3, gat2_dropRate = 0.3, gat1_channel = 3, gat1_nhead = 3, gat2_channel = 6, pool_channel = 12, dense_channel = 6)



#model.summary()

In [8]:

def callback_saveweight_train(model , name, result_tr, current_tr_loss=np.inf, current_tr_acc=0,current_tr_f1=0,current_tr_ROCauc=0,logoutput=None):
    new_tr_loss=result_tr[0]
    new_tr_acc=result_tr[1]
    new_tr_f1 = result_tr[2]
    new_tr_ROCauc=result_tr[3]
    if new_tr_loss<=current_tr_loss:
        if new_tr_loss==current_tr_loss and new_tr_acc>=current_tr_acc:
            model.save_weights(name+"_tr_loss")
            print("train_loss improve from "+str(current_tr_loss)+" to "+str(new_tr_loss) )
            logoutput.write("train_loss improve from "+str(current_tr_loss)+" to "+str(new_tr_loss)+"\n")
        if new_tr_loss<current_tr_loss:
            model.save_weights(name+"_tr_loss")
            print("train_loss improve from "+str(current_tr_loss)+" to "+str(new_tr_loss) )
            logoutput.write("train_loss improve from "+str(current_tr_loss)+" to "+str(new_tr_loss)+"\n")
    else:
        print("train_loss doesnot improve from "+str(current_tr_loss))
        logoutput.write("train_loss doesnot improve from "+str(current_tr_loss)+"\n")
    
    if new_tr_f1>=current_tr_f1:
        model.save_weights(name+"_tr_f1")
        print("tr_f1 improve from "+str(current_tr_f1)+" to "+str(new_tr_f1) )
        logoutput.write("tr_f1 improve from "+str(current_tr_f1)+" to "+str(new_tr_f1)+"\n")
    else:
        print("tr_f1 doesnot improve from "+str(current_tr_f1))
        logoutput.write("tr_f1 doesnot improve from "+str(current_tr_f1)+"\n")
    
    if new_tr_acc>=current_tr_acc:
        #print("va_f1 improve from "+str(current_va_f1)+" to "+str(new_va_f1) )
        if new_tr_acc==current_tr_acc and new_tr_loss<=current_tr_loss:
            model.save_weights(name+"_tr_acc")
        if new_tr_acc>current_tr_acc:
            model.save_weights(name+"_tr_acc")
    
    if new_tr_ROCauc>=current_tr_ROCauc:
        model.save_weights(name+"_tr_ROCauc")
    
    if new_tr_ROCauc>=current_tr_ROCauc and new_tr_loss<=current_tr_loss:
        model.save_weights(name+"_tr_ROCauc_loss")
    
    if new_tr_loss<=current_tr_loss:
        current_tr_loss=new_tr_loss
    if new_tr_acc>=current_tr_acc:
        current_tr_acc=new_tr_acc
    if new_tr_f1>=current_tr_f1:
        current_tr_f1=new_tr_f1
    
    return current_tr_loss,current_tr_acc,current_tr_f1,current_tr_ROCauc


In [9]:
def callback_saveweight(model , name, result_va, result_te, current_va_loss=np.inf, current_va_acc=0,
                        current_te_loss=np.inf, current_te_acc=0, current_va_f1=0, current_va_ROCauc=0,logoutput=None):
    new_va_loss=result_va[0]
    new_va_acc=result_va[1]
    new_te_loss=result_te[0]
    new_te_acc=result_te[1]
    new_va_f1=result_va[2]
    new_va_ROCauc=result_va[3]
    if new_va_loss<=current_va_loss:
        if new_va_loss==current_va_loss and new_va_acc>=current_va_acc:
            model.save_weights(name+"_va_loss")
        if new_va_loss<current_va_loss:
            model.save_weights(name+"_va_loss")

    if new_va_acc>=current_va_acc:
        if new_va_acc==current_va_acc and new_va_loss<=current_va_loss:
            model.save_weights(name+"_va_acc")
        if new_va_acc>current_va_acc:
            model.save_weights(name+"_va_acc")
        
    if new_te_loss<=current_te_loss:
        if new_te_loss==current_te_loss and new_te_acc>=current_te_acc:
            model.save_weights(name+"_te_loss")
        if new_te_loss<current_te_loss:
            model.save_weights(name+"_te_loss")
        
    if new_te_acc>=current_te_acc:
        if new_te_acc==current_te_acc and new_te_loss<=current_te_loss:
            model.save_weights(name+"_te_acc")
        if new_te_acc>current_te_acc:
            model.save_weights(name+"_te_acc")
    
    # va_acc_gain=abs((new_va_acc-current_va_acc)/current_va_acc)
    # va_loss_gain=abs((new_va_loss-current_va_loss)/current_va_loss)
    # if new_va_acc>=current_va_acc and new_va_loss<=current_va_loss:
    #     model.save_weights(name+"_va_accLoss")
    # if new_va_acc>=current_va_acc and new_va_loss>current_va_loss:
    #     if va_acc_gain>=va_loss_gain:
    #         model.save_weights(name+"_va_accLoss")
            
    if new_va_f1>=current_va_f1:
        model.save_weights(name+"_va_f1")
        print("va_f1 improve from "+str(current_va_f1)+" to "+str(new_va_f1) )
        logoutput.write("va_f1 improve from "+str(current_va_f1)+" to "+str(new_va_f1)+"\n")
    else:
        print("va_f1 doesnot improve from "+str(current_va_f1))
        logoutput.write("va_f1 doesnot improve from "+str(current_va_f1)+"\n")
    
    if new_va_ROCauc>=current_va_ROCauc:
        model.save_weights(name+"_va_ROCauc")
    
    if new_va_ROCauc>=current_va_ROCauc and new_va_loss<=current_va_loss:
        model.save_weights(name+"_va_ROCauc_loss")
    
    if new_va_loss<=current_va_loss:
        current_va_loss=new_va_loss
    if new_va_acc>=current_va_acc:
        current_va_acc=new_va_acc
    if new_te_loss<=current_te_loss:
        current_te_loss=new_te_loss
    if new_te_acc>=current_te_acc:
        current_te_acc=new_te_acc
    if new_va_f1>=current_va_f1:
        current_va_f1=new_va_f1
    if new_va_ROCauc>=current_va_ROCauc:
        current_va_ROCauc=new_va_ROCauc
    
    return current_va_loss,current_va_acc,current_te_loss,current_te_acc,current_va_f1,current_va_ROCauc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()
# Training function
# @tf.function



In [10]:
import sklearn
'''
def evaluate_all(data, model,cutoff=0.5):
    x=data['x']
    y=data['y']
    info=data['info']
    cols=data['cols']
    a=data['pathway_a']
    
    predictions = model([x,a],training=False)
    loss=float(loss_fn(y, predictions).numpy())
    roc_auc_score=sklearn.metrics.roc_auc_score(y[:,1], predictions[:,1])
    acc=sklearn.metrics.accuracy_score(y[:,1], predictions[:,1]>cutoff)
    f1=sklearn.metrics.f1_score(y[:,1],predictions[:,1]>cutoff)
    return loss,acc,f1,roc_auc_score
'''
def evaluate_all(loader, model,cutoff=0.5):
    step = 0
    prediction_list=[]
    label_list=[]
    for batch in loader:
        step += 1
        inputs, target = batch
        predictions = model(inputs,training=False)
        # predictions=predictions.numpy().reshape([batch_size,-1])
        prediction_list.extend(predictions.numpy())
        label_list.extend(target)
        if step == loader.steps_per_epoch:
            break
    
    label_list=np.asarray(label_list)
    prediction_list = np.asarray(prediction_list)
    loss=float(loss_fn(label_list, prediction_list).numpy())
    roc_auc_score=sklearn.metrics.roc_auc_score(label_list[:,1], prediction_list[:,1])
    acc=sklearn.metrics.accuracy_score(label_list[:,1], prediction_list[:,1]>cutoff)
    f1=sklearn.metrics.f1_score(label_list[:,1],prediction_list[:,1]>cutoff)
    return loss,acc,f1,roc_auc_score




In [11]:
from sklearn.utils import resample
import random

def bootstp3(rx,ry,fold,rand_state):
    random.seed(rand_state)
    n_zero=int(sum(ry)[0])
    n_one=int(sum(ry)[1])
    n=max(n_zero, n_one)*fold
    dif0=n-n_zero
    dif1=n-n_one
    ind_one=ry[:,1]>0
    sam1=rx[ind_one]
    bootx1= resample(sam1, replace=True, n_samples=dif1)
    ind_zero=ry[:,0]>0
    sam0=rx[ind_zero]
    bootx0= resample(sam0, replace=True, n_samples=dif0)
    newx=np.concatenate((bootx0,sam0,bootx1,sam1),axis=0)
    a=np.zeros([n]).reshape(n,1)
    b=np.ones([n]).reshape(n,1)
    booty1=np.concatenate((a,b),axis=1)
    booty0=np.concatenate((b,a),axis=1)
    newy=np.concatenate((booty0,booty1),axis=0)
    return [newx,newy]



In [12]:



import os

def train_process(patience, direc, loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation, epochs,model,train_data,val_data,test_data,skipimbalanceratio=0.7,loss_fn=tfa.losses.SigmoidFocalCrossEntropy(reduction=tf.keras.losses.Reduction.AUTO)):
    # Setup training
    if not os.path.exists(direc):
        os.makedirs(direc)
    
    logoutput=open(direc+"_log.txt",'w')
    optimizer = Adam(learning_rate=0.0001)
    #loss_fn = CategoricalCrossentropy()
    def train_on_batch(inputs, target, model,loss_fn):
        with tf.GradientTape() as tape:
            # target=target.reshape([-1,1,2])
            predictions = model(inputs,training=True)
            #predictions=predictions.numpy().reshape([batch_size,-1])
            loss = loss_fn(target, predictions) + sum(model.losses)
            #loss = loss_fn(target, predictions)
            acc = tf.reduce_mean(categorical_accuracy(target, predictions))
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss, acc
    
    best_val_loss = 9999999999
    best_val_acc = 0
    best_val_f1=0
    current_patience = patience
    step = 0
    numepoch=0
    # Training loop
    results_tr = []
    for batch in loader_tr:
        # Training step
        inputs, target = batch
        
        if skipimbalanceratio !=1:
           one_zero=sum(target)
           if one_zero[0]>=skipimbalanceratio*(one_zero[0]+one_zero[1]):
               print("\n########at Epoch "+str(numepoch)+" skip one_zero 0 >0.7\n")
           if one_zero[1]>=skipimbalanceratio*(one_zero[0]+one_zero[1]):
               print("\n########at Epoch "+str(numepoch)+" skip one_zero 1 >0.7\n")
           else:
               loss, acc = train_on_batch(inputs, target, model,loss_fn)
               step += 1
        else:
               loss, acc = train_on_batch(inputs, target, model,loss_fn)
               step += 1
        #results_tr.append((loss, acc, len(target)))
        if step == steps_evaluation:#loader_tr.steps_per_epoch:
            #results_va = evaluate(loader_va, model)
            results_va = evaluate_all(loader_va, model)
            #results_tr = evaluate(loader_tr,model)
            results_tr = evaluate_all(loader_tr,model)
            if results_va[2] >= best_val_f1:
                best_val_f1 = results_va[2]
                current_patience = patience
                #results_te = evaluate(loader_te, model)
                results_te=evaluate_all(loader_te, model)
            else:
                current_patience -= 1
                if current_patience == 0:
                    print("Early stopping")
                    break
            
            # Print results
            #results_tr = np.array(results_tr)
            #results_tr = np.average(results_tr[:, :-1], 0, weights=results_tr[:, -1])
        if step == steps_per_epoch:#loader_tr.steps_per_epoch:
            print(
                "Epoch:{:d}, Train loss:{:.4f},acc:{:.4f},f1:{:.4f},ROCauc:{:.4f}| "
                "Valid loss:{:.4f},acc:{:.4f},f1:{:.4f},ROCauc:{:.4f}|"
                "Test loss:{:.4f},acc:{:.4f},f1:{:.4f},ROCauc:{:.4f}".format(
                    numepoch,*results_tr, *results_va, *results_te
                )
            )
            logoutput.write(
                "Epoch:{:d}, Train loss:{:.4f},acc:{:.4f},f1:{:.4f},ROCauc:{:.4f}| "
                "Valid loss:{:.4f},acc:{:.4f},f1:{:.4f},ROCauc:{:.4f}|"
                "Test loss:{:.4f},acc:{:.4f},f1:{:.4f},ROCauc:{:.4f}".format(
                    numepoch,*results_tr, *results_va, *results_te
                )
            )
            logoutput.write("\n")
            numepoch+=1
            model.save_weights(direc+"_current_minibatch")
            if numepoch==epochs:
                break
            
            if numepoch==1:
                current_va_loss,current_va_acc,current_te_loss,current_te_acc,current_va_f1,current_va_ROCauc=callback_saveweight(model, direc, results_va, results_te,logoutput=logoutput)
                current_tr_loss,current_tr_acc,current_tr_f1,current_tr_ROCauc=callback_saveweight_train(model,direc,results_tr,logoutput=logoutput)
            else:
                current_va_loss,current_va_acc,current_te_loss,current_te_acc,current_va_f1,current_va_ROCauc=callback_saveweight(model, direc, results_va, results_te,current_va_loss, current_va_acc, current_te_loss,current_te_acc, current_va_f1, current_va_ROCauc,logoutput=logoutput)
                
                current_tr_loss,current_tr_acc,current_tr_f1,current_tr_ROCauc=callback_saveweight_train(model,direc,results_tr,current_tr_loss,current_tr_acc,current_tr_f1,current_tr_ROCauc,logoutput=logoutput)
            # Reset epoch
            results_tr = []
            step = 0




In [ ]:
# 5-fold pre-train merge data

import random
import numpy as np
random.seed(996)

"""
datafromnpz1=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_SKCM_2y_zscore.npz')
datafromnpz2=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_BLCA_2y_zscore.npz')
datafromnpz3=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_STAD_zscore.npz')

datafromnpz = {}
datafromnpz['info']=np.concatenate([datafromnpz1['info'],datafromnpz2['info'],datafromnpz3['info']],axis=0)
datafromnpz['x']=np.concatenate([datafromnpz1['x'],datafromnpz2['x'],datafromnpz3['x']],axis=0)
datafromnpz['y']=np.concatenate([datafromnpz1['y'],datafromnpz2['y'],datafromnpz3['y']],axis=0)
datafromnpz['cols']=datafromnpz1['cols']
datafromnpz['pathway_a']=datafromnpz1['pathway_a']

#datafromnpz=datafromnpz1
"""
#datafromnpz = np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_ALLPrj_zscore.npz')
datafromnpz=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_STAD_zscore.npz')

ind=[]
for i in range(len(datafromnpz['info'])):
    ind.append(i)

rind=random.sample(ind,len(ind))
rx=datafromnpz['x'][rind]
ry=datafromnpz['y'][rind]
ry=np.array(ry,dtype='int32')
rinfo=datafromnpz['info'][rind]


hyperParams = np.array([
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,20],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,20],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,20],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,20],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,20]
])

epochs = 400  # Number of training epochs
patience = 400  # Patience for early stopping


import math
from sklearn.model_selection import StratifiedKFold
np.random.seed(996)


fold=0
skf = StratifiedKFold(n_splits=5)
y_binary = ry[:,1]
for train, test in skf.split(rx,y_binary):
    
    # # Uncomment if rename error occurs
    #if fold==0:                  
    #     fold+=1
    #     continue
    
    runtimes = 0
    print("########### fold: "+str(fold)+" ############")
    # nsam=math.floor(0.25*len(train))
    # train_val=random.sample(list(train),nsam)
    # train_train=list(set(train)-set(train_val))
    #print("%s %s" % (train, test))
    print(sum(ry[test]))
    
    newdata_train={}
    newdata_train['x']=rx[train]
    newdata_train['y']=ry[train]
    newdata_train['info']=rinfo[train]
    newdata_train['cols']=datafromnpz['cols']
    newdata_train['pathway_a']=datafromnpz['pathway_a']
    
    print("Before bootstrap:")
    print(sum(newdata_train['y']))
    [newdata_train['x'],newdata_train['y']]=bootstp(newdata_train['x'],newdata_train['y'])
    print("After bootstrap:")
    print(sum(newdata_train['y']))
    
    # newdata_val={}
    # newdata_val['x']=rx[train_val]
    # newdata_val['y']=ry[train_val]
    # newdata_val['info']=rinfo[train_val]
    # newdata_val['cols']=datafromnpz['cols']
    # newdata_val['pathway_a']=datafromnpz['pathway_a']
    
    newdata_test={}
    newdata_test['x']=rx[test]
    newdata_test['y']=ry[test]
    newdata_test['info']=rinfo[test]
    newdata_test['cols']=datafromnpz['cols']
    newdata_test['pathway_a']=datafromnpz['pathway_a']

    data_train=MyDataset(newdata_train,transforms=NormalizeAdj())
    # data_val=MyDataset(newdata_val,transforms=NormalizeAdj())
    data_test=MyDataset(newdata_test,transforms=NormalizeAdj())
    
    data_train.a=sp_matrix_to_sp_tensor(data_train.a)
    # data_val.a=sp_matrix_to_sp_tensor(data_val.a)
    data_test.a=sp_matrix_to_sp_tensor(data_test.a)
    
    batch_size=int(hyperParams[fold,12])
        
    sample_size = sum(sum(newdata_train['y']))
    steps_per_epoch=int(np.ceil(sample_size/batch_size))
    
    #loader_tr = MixedLoader(data_train, batch_size=batch_size, epochs=epochs,shuffle=True)
    loader_tr = MixedLoader(data_train, batch_size=batch_size, epochs=None,shuffle=True)
    # loader_va = MixedLoader(data_val, batch_size=len(data_val),shuffle=False)
    loader_te = MixedLoader(data_test, batch_size=len(data_test),shuffle=False)
    
    [mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, gat1_dropRate, 
     gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel]=hyperParams[fold,:-1]
    print('hyper-parameters:')
    for i in [mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, gat1_dropRate, 
     gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel]:
        print(i)
    
    model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                    gat1_dropRate, gat2_dropRate, int(gat1_channel), int(gat1_nhead), int(gat2_channel), int(pool_channel), int(dense_channel))
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/merge_SKCM_BLCA_STAD_randParam_noboostp"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=1)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/merge_SKCM_BLCA_STAD_randParam_boostp07"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=0.7)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/SKCM_randParam_boostp07"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=0.7)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/SKCM_randParam_boostp1_"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=1)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/merge_SKCM_BLCA_STAD_randParam_boostp1_"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=1)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/merge_TCGAall_randParam_boostp1_"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=1)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/STAD_randParam_boostp1_"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=1)
    #train_process(patience, "./weights_counts_zscore/TCGA_pretrain/BLCA_randParam_boostp1_"+str(fold)+'/', loader_tr, loader_te, loader_te,steps_per_epoch,epochs, model,newdata_train,newdata_test,newdata_test,skipimbalanceratio=1)
    fold+=1

In [ ]:
## Transfer 5-fold CV

import random
random.seed(996)

#datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Gide_zscore.npz') #SKCM (merged better for models used TCGA data)
#datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Liu_zscore.npz') #SKCM (merged better for models used TCGA data)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_IMvigor_zscore.npz') #BLCA (merge_boostp1 better)
#datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Kim_zscore.npz') #STAD (SKCM better?)
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_GSE176307_zscore.npz') 
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_Mariathasan_zscore.npz') 
datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_Checkmate025_Niv_zscore.npz')



loss_fn = tfa.losses.SigmoidFocalCrossEntropy(gamma=2,reduction=tf.keras.losses.Reduction.AUTO) #alpha>0.5 works for more loss on positive data!
ind=[]
for i in range(len(datafromnpz['info'])):
    ind.append(i)

rind=random.sample(ind,len(ind))
rx=datafromnpz['x'][rind]
ry=datafromnpz['y'][rind]
rinfo=datafromnpz['info'][rind]



###  random parameter  ###
mapping_l1_reg = 2.5e-3
gat1_l2_reg = 2.5e-3
gat2_l2_reg = 2.5e-3
pool_l1_reg = 2.5e-3
x_dropRate = 0.5
gat1_dropRate = 0.4
gat2_dropRate = 0.4
gat1_channel = 4
gat1_nhead = 4
gat2_channel = 4
pool_channel = 8
dense_channel = 8
batch_size = 20  # Batch size


###  opti  ###
# mapping_l1_reg = 0.00009321777193822189
# gat1_l2_reg = 0.006330908565823178
# gat2_l2_reg = 0.0025484322376082657
# pool_l1_reg = 0.006500277466013919
# x_dropRate = 0.49729938752948555
# gat1_dropRate = 0.15052736244190676
# gat2_dropRate = 0.3243055268545104
# gat1_channel = 6
# gat1_nhead = 2
# gat2_channel = 6
# pool_channel = 11
# dense_channel = 3
# batch_size = 7  # Batch size
epochs = 400  # Number of training epochs
patience = 400  # Patience for early stopping
import math
from sklearn.model_selection import StratifiedKFold,train_test_split
np.random.seed(996)
fold=0
skf = StratifiedKFold(n_splits=5)
y_binary = ry[:,1]
for train, test in skf.split(rx,y_binary):
    
    # # Uncomment if rename error occurs
    if fold<=2:                  
         fold+=1
         continue
    
    runtimes = 0
    print("########### fold: "+str(fold)+" ############")
    # nsam=math.floor(0.25*len(train))
    # train_val=random.sample(list(train),nsam)
    # train_train=list(set(train)-set(train_val))
    print("%s %s" % (train, test))
    print(sum(ry[test]))
    
    
    newdata_train={}
    newdata_val={}
    newdata_train['cols']=datafromnpz['cols']
    newdata_train['pathway_a']=datafromnpz['pathway_a']
    newdata_val['cols']=datafromnpz['cols']
    newdata_val['pathway_a']=datafromnpz['pathway_a']
    newdata_train['x'],newdata_val['x'], newdata_train['y'], newdata_val['y']=train_test_split(rx[train],ry[train],train_size=0.8, random_state=996, shuffle=True, stratify=ry[train])
    #newdata_train['x']=rx[train]
    #newdata_train['y']=ry[train]
    newdata_train['info']=newdata_train['y'] #rinfo[train]
    newdata_val['info']=newdata_val['y']
    
    print("before bootstp")
    print(sum(newdata_train['y']))
    print(sum(newdata_val['y']))
    #sample_size = sum(sum(newdata_train['y'])) # for Liu only before bootstp3
    #steps_per_epoch=int(np.ceil(sample_size/batch_size))
    steps_evaluation=5 #int(np.ceil(sum(sum(newdata_train['y']))/batch_size))
    print("steps_evaluation="+str(steps_evaluation))
    #[newdata_train['x'],newdata_train['y']]=bootstp(newdata_train['x'],newdata_train['y'])
    [newdata_train['x'],newdata_train['y']]=bootstp3(newdata_train['x'],newdata_train['y'],2,fold)
    sample_size = sum(sum(newdata_train['y']))
    steps_per_epoch=int(np.ceil(sample_size/batch_size))
    print("steps_per_epoch="+str(steps_per_epoch))
    if steps_per_epoch < steps_evaluation:
        steps_evaluation=steps_per_epoch
    print("reset steps_evaluation="+str(steps_evaluation))
    """
    bootx_list=[]
    booty_list=[]
    for _ in range(5):
       #bootx,booty=bootstp_random(newdata_train['x'],newdata_train['y'])
       bootx,booty=bootstp(newdata_train['x'],newdata_train['y'])
       bootx_list.append(bootx)
       booty_list.append(booty)
    
    newdata_train['x']=np.concatenate(bootx_list,axis=0)
    newdata_train['y']=np.concatenate(booty_list,axis=0)
    """

    print(sum(newdata_train['y']))
    ind=list(np.arange(len(newdata_train['x'])))
    rind=random.sample(ind,len(ind))
    rind=random.sample(rind,len(ind))
    rind=random.sample(rind,len(ind))
    newdata_train['x']=newdata_train['x'][rind]
    newdata_train['y']=newdata_train['y'][rind]
    
    newdata_test={}
    newdata_test['x']=rx[test]
    newdata_test['y']=ry[test]
    newdata_test['info']=rinfo[test]
    newdata_test['cols']=datafromnpz['cols']
    newdata_test['pathway_a']=datafromnpz['pathway_a']
    
    data_train=MyDataset(newdata_train,transforms=NormalizeAdj())
    data_val=MyDataset(newdata_val,transforms=NormalizeAdj())
    data_test=MyDataset(newdata_test,transforms=NormalizeAdj())
    
    data_train.a=sp_matrix_to_sp_tensor(data_train.a)
    data_val.a=sp_matrix_to_sp_tensor(data_val.a)
    data_test.a=sp_matrix_to_sp_tensor(data_test.a)
    
    #loader_tr = MixedLoader(data_train, batch_size=batch_size, epochs=epochs,shuffle=True)
    loader_tr = MixedLoader(data_train, batch_size=batch_size, epochs=None,shuffle=True)
    loader_te = MixedLoader(data_test, batch_size=len(data_test),shuffle=False)
    loader_va = MixedLoader(data_val, batch_size=len(data_test),shuffle=False)
    
    model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                gat1_dropRate, gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel)
    
    model.load_weights("./weights_counts_zscore/TCGA_pretrain/merge_SKCM_BLCA_STAD_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/TCGA_pretrain/SKCM2y_randParam_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/TCGA_pretrain/BLCA_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/TCGA_pretrain/SKCM_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/STAD_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore_focaloss/TCGA_pretrain/merge_SKCM_BLCA_STAD_bootstp3_5_steval5_"+str(fold)+"/_va_f1")

    #for Liu the sample_size is before bootstp3 is wrong
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val_run2/Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Kim_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval4_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    # train_process(patience,"./weights_counts_zscore_focaloss_val_run2/IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    # train_process(patience,"./weights_counts_zscore_focaloss_val/GSE176307_zscore_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    # train_process(patience,"./weights_counts_zscore_focaloss_val/Mariathasan_zscore_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    train_process(patience,"./weights_counts_zscore_focaloss_val/Checkmate025_Niv_zscore_bsp_TCGA3boostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)

    fold+=1



In [12]:
# within study evaluation

from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
import numpy as np
import sklearn
import random
random.seed(996)


###  random parameter  ###
mapping_l1_reg = 2.5e-3
gat1_l2_reg = 2.5e-3
gat2_l2_reg = 2.5e-3
pool_l1_reg = 2.5e-3
x_dropRate = 0.5
gat1_dropRate = 0.4
gat2_dropRate = 0.4
gat1_channel = 4
gat1_nhead = 4
gat2_channel = 4
pool_channel = 8
dense_channel = 8
batch_size = 10  # Batch size
hyperParams_random = np.array([
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,10],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,10],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,10],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,10],
[2.5e-3,2.5e-3,2.5e-3,2.5e-3,0.5,0.4,0.4,4,4,4,8,8,10]
])


###  opti IMvigor ROC  ###
hyperParams_IMvigor=np.array([
[0.00675190391336071,0.005493762552723812,0.004519753059018278,0.0045195975667924534,0.46362793709225414,0.48819564127373993,0.2649679082634909,7,9,5,25,11,9],
[0.004753781661814064,0.00984774073557786,0.0048767374957450465,0.0007873736162618307,0.3194759952137326,0.30398181418512144,0.4979219375485108,3,3,10,12,9,5],
[0.0071387418189729525,0.005222498035643917,0.0027004909388650775,0.0009363385112361626,0.29024065711708774,0.14865279095047879,0.15748988511469506,3,6,6,18,13,6],
[0.003968517903566386,0.004080120823919732,0.0042587618179442885,0.009978974683794863,0.20088388535107163,0.3332139513019352,0.10333163146773422,4,4,12,5,5,3],
[0.0018638222878056294,0.0029886076626495254,0.0040239676839539254,0.003992625256227416,0.18392455882921968,0.23979599005221788,0.2249788516873279,3,5,15,13,7,4]
])

###  opti Liu ROC  ###
hyperParams_Liu=np.array([
[0.009161283552587185,0.0086139933888295,0.0010692657989040144,0.003913069210703885,0.3748124481558166,0.49048942098340775,0.4925881321925005,4.0,9.0,8.0,14.0,14.0,3],
[0.009161283552587185,0.0086139933888295,0.0010692657989040144,0.003913069210703885,0.3748124481558166,0.49048942098340775,0.4925881321925005,4.0,9.0,8.0,14.0,14.0,3],
[0.009161283552587185,0.0086139933888295,0.0010692657989040144,0.003913069210703885,0.3748124481558166,0.49048942098340775,0.4925881321925005,4.0,9.0,8.0,14.0,14.0,3],
[0.009161283552587185,0.0086139933888295,0.0010692657989040144,0.003913069210703885,0.3748124481558166,0.49048942098340775,0.4925881321925005,4.0,9.0,8.0,14.0,14.0,3],
[0.009161283552587185,0.0086139933888295,0.0010692657989040144,0.003913069210703885,0.3748124481558166,0.49048942098340775,0.4925881321925005,4.0,9.0,8.0,14.0,14.0,3]
])



model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                    gat1_dropRate, gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel)


cutrange=np.arange(0.05,0.95,0.05)
def find_cutoff(datafromnpz,weight_folder,weight_presuffix,weight_metric,hyper):
    ind=[]
    for i in range(len(datafromnpz['info'])):
        ind.append(i)
    rind=random.sample(ind,len(ind))
    rx=datafromnpz['x'][rind]
    ry=datafromnpz['y'][rind]
    rinfo=datafromnpz['info'][rind]
    
    skf = StratifiedKFold(n_splits=5)
    fold=0
    y_binary = ry[:,1]
    result=np.zeros([5,6,cutrange.shape[0]])
    for train, test in skf.split(rx,y_binary):
            x=rx[test]
            y=ry[test]
            info=rinfo[test]
            cols=datafromnpz['cols']
            a=datafromnpz['pathway_a']
            
            if hyper=='IMvigor':
                hyperParams=hyperParams_IMvigor
            if hyper=='random':
                hyperParams=hyperParams_random
            if hyper=='Liu':
                hyperParams=hyperParams_Liu
            
            [mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, gat1_dropRate, 
             gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel]=hyperParams[fold,:-1]
            print('hyper-parameters:')
            for i in [mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, gat1_dropRate, 
                      gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel]:
                print(i)
            model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                              gat1_dropRate, gat2_dropRate, int(gat1_channel), int(gat1_nhead), int(gat2_channel), int(pool_channel), int(dense_channel))
            
            ncut=0
            for cutoff in cutrange:
                model.load_weights(weight_folder+weight_presuffix+str(fold)+'/_'+weight_metric)
                predictions = model([x,a],training=False)
                [pre, rec, fscore,useless] = precision_recall_fscore_support(y[:,1],predictions[:,1]>=cutoff,average='binary')
                mcc = sklearn.metrics.matthews_corrcoef(y[:,1],predictions[:,1]>=cutoff)
                acc=sklearn.metrics.accuracy_score(y[:,1], predictions[:,1]>cutoff)
                f1=sklearn.metrics.f1_score(y[:,1],predictions[:,1]>cutoff)
                result[fold,:,ncut]=[pre,rec,fscore,mcc,acc,f1]
                ncut+=1
            fold+=1
    return result

def get_scores(datafromnpz,weight_folder,weight_presuffix,weight_metric,cutoff,hyper):
    random.seed(996)
    ind=[]
    for i in range(len(datafromnpz['info'])):
        ind.append(i)
    rind=random.sample(ind,len(ind))
    rx=datafromnpz['x'][rind]
    ry=datafromnpz['y'][rind]
    rinfo=datafromnpz['info'][rind]
    
    allfold_ROCauc=[]
    allfold_f1=[]
    allfold_acc=[]
    random.seed(996)
    skf = StratifiedKFold(n_splits=5)
    fold=0
    y_binary = ry[:,1]
    for train, test in skf.split(rx,y_binary):
            #print('fold '+str(fold)+' : ')
            #print("%s" % (test))
            x=rx[test]
            y=ry[test]
            #x=rx[train]
            #y=ry[train]
            info=rinfo[test]
            cols=datafromnpz['cols']
            a=datafromnpz['pathway_a']
            #print("%s" % (info))
            #print("%s" % (y[:,1]))
            print(sum(y))
            if hyper=='IMvigor':
                hyperParams=hyperParams_IMvigor
            if hyper=='random':
                hyperParams=hyperParams_random
            if hyper=='Liu':
                hyperParams=hyperParams_Liu
            
            [mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, gat1_dropRate, 
             gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel]=hyperParams[fold,:-1]
            #print('hyper-parameters:')
            #for i in [mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, gat1_dropRate, 
            #          gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel]:
            #    print(i)
            model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                              gat1_dropRate, gat2_dropRate, int(gat1_channel), int(gat1_nhead), int(gat2_channel), int(pool_channel), int(dense_channel))
            
            results=np.zeros([1,y.shape[0],y.shape[1]])
            #for weight_metric in ['tr_f1','tr_f1']:
            #for weight_metric in ['tr_ROCauc','tr_ROCauc']:
            for weight_metric in ['va_f1','va_f1']:
            #for weight_metric in ['va_ROCauc','va_ROCauc']:
            #for weight_metric in ['te_loss','te_loss']:
                print("load weights from "+weight_presuffix+str(fold)+'/_'+weight_metric)
                model.load_weights(weight_folder+weight_presuffix+str(fold)+'/_'+weight_metric)
                predictions = model([x,a],training=False)
                results+=predictions
            
            results_mean = np.mean(results, axis=0)/2
            roc_auc_score=sklearn.metrics.roc_auc_score(y[:,1], results_mean[:,1])
            acc=sklearn.metrics.accuracy_score(y[:,1], results_mean[:,1]>cutoff)
            f1=sklearn.metrics.f1_score(y[:,1],results_mean[:,1]>cutoff)
            #print("%s" % (predictions[:,1]>cutoff))
            print('acc = '+str(acc))
            print('f1 score = '+str(f1))
            print('roc_auc_score = '+str(roc_auc_score))
            allfold_ROCauc.append(roc_auc_score)
            allfold_f1.append(f1)
            allfold_acc.append(acc)
            fold+=1
    return [allfold_ROCauc, allfold_f1, allfold_acc]
        

    
    
    
random.seed(996)
#datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Gide_zscore.npz')
#datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Auslander_zscore.npz') # (SKCM)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Gide_zscore.npz') 
datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_IMvigor_zscore.npz') #BLCA (merge_boostp1 better)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Kim_zscore.npz') #STAD (SKCM better?)
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_GSE176307_zscore.npz') 
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_Mariathasan_zscore.npz') 
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_Checkmate025_Niv_zscore.npz')



#TCGA
#datafromnpz=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_SKCM_2y_zscore.npz')#merge_SKCM_BLCA_STAD_randParam_boostp1_​ tr better
#datafromnpz=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_BLCA_2y_zscore.npz') #BLCA_randParam_boostp1_ tr best
#datafromnpz=np.load('immunotherapy_tcga_onehotlabel_kegg_pathgraph_STAD_zscore.npz')


# result=find_cutoff(datafromnpz,'../tem/','Gide_bsp_randParam_','va_f1','random')
# result_avg=np.average(result,axis=0)
# import matplotlib.pyplot as plt
# # create data
# cutrange=np.arange(0.05,0.95,0.05)
# pre=result_avg[0,:]
# rec=result_avg[1,:]
# fscore=result_avg[2,:]
# mcc=result_avg[3,:]
# acc=result_avg[4,:]
# # plot lines
# plt.plot(cutrange, pre, label = "precision")
# plt.plot(cutrange, rec, label = "recall")
# plt.plot(cutrange, fscore, label = "fscore")
# plt.plot(cutrange, mcc, label = "mcc")
# plt.plot(cutrange, acc, label = "acc")
# plt.legend()
# plt.show()


random.seed(996)
cutoff=0.5
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','Gide_TCGAtrans_randParam_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','Gide_TCGAtrans_randParam_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','SKCM2y_randParam_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','Gide_TCGAtrans_randParam_','current_minibatch',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','merge_SKCM_BLCA_STAD_randParam_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/TCGA_pretrain/','SKCM2y_randParam_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_boostp07','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','merge_SKCM_BLCA_STAD_randParam_nobootsrp','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/TCGA_pretrain/','merge_SKCM_BLCA_STAD_randParam_boostp1_','tr_f1',cutoff,'random')
#print("acc mean = "+str(np.mean(allfold_acc)))
#print("f1 mean = "+str(np.mean(allfold_f1)))
#print("ROCauc mean = "+str(np.mean(allfold_ROCauc)))
#print("----")
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_boostp1_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','BLCA_randParam_boostp1_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','Liu_TCGAtrans_randParam_bulk4_bootstp1_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp1_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','merge_TCGAall_randParam_boostp1_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','IMvigor_bsp_TCGAtrans_randParam_bulk4_bootstp1_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_','tr_f1',cutoff,'random')


#print("acc mean = "+str(np.mean(allfold_acc)))
#print("f1 mean = "+str(np.mean(allfold_f1)))
#print("ROCauc mean = "+str(np.mean(allfold_ROCauc)))
print("---")
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_5fold_val/','Liu_bsp_TCGAtransSKCMBLCASTAD_bulk4_bootstp1_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp3_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_','tr_f1',cutoff,'random')
# [allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Kim_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','tr_f1',cutoff,'random')
#[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val_run2/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','tr_f1',cutoff,'random')
# [allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Kim_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval4_','tr_f1',cutoff,'random')
[allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','tr_f1',cutoff,'random')
# [allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val_run2/','IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','tr_f1',cutoff,'random')
# [allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','GSE176307_zscore_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff,'random')
# [allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Mariathasan_zscore_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff,'random')
# [allfold_ROCauc, allfold_f1, allfold_acc] = get_scores(datafromnpz,'./weights_counts_zscore_focaloss_val/','Checkmate025_Niv_zscore_bsp_TCGA3boostp1_bootstp3_2_steval5_','va_f1',cutoff,'random')

    

print("acc mean = "+str(np.mean(allfold_acc)))
print("f1 mean = "+str(np.mean(allfold_f1)))
print("ROCauc mean = "+str(np.mean(allfold_ROCauc)))






---
[34.  5.]
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_0/_va_f1
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_0/_va_f1
acc = 0.8461538461538461
f1 score = 0.0
roc_auc_score = 0.5470588235294117
[34.  5.]
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_1/_va_f1
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_1/_va_f1
acc = 0.717948717948718
f1 score = 0.26666666666666666
roc_auc_score = 0.7
[33.  5.]
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_2/_va_f1
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_2/_va_f1
acc = 0.7894736842105263
f1 score = 0.3333333333333333
roc_auc_score = 0.7636363636363637
[33.  5.]
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_3/_va_f1
load weights from IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_3/_va_f1
acc = 0.842105263157

In [91]:
# cross study evaluation

from sklearn.model_selection import KFold
import numpy as np
import sklearn
import random
from random import choices
random.seed(996)


###  random parameter  ###
mapping_l1_reg = 2.5e-3
gat1_l2_reg = 2.5e-3
gat2_l2_reg = 2.5e-3
pool_l1_reg = 2.5e-3
x_dropRate = 0.5
gat1_dropRate = 0.4
gat2_dropRate = 0.4
gat1_channel = 4
gat1_nhead = 4
gat2_channel = 4
pool_channel = 8
dense_channel = 8
batch_size = 10  # Batch size

model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                    gat1_dropRate, gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel)



    

def get_scores(datafromnpz,weight_folder,weight_presuffix,weight_metric,cutoff):
            x=datafromnpz['x']
            y=datafromnpz['y']
            info=datafromnpz['info']
            cols=datafromnpz['cols']
            a=datafromnpz['pathway_a']
            # print("%s" % (info))
            #print("%s" % (y[:,1]))
            model.load_weights(weight_folder+weight_presuffix+'/_'+weight_metric)
            predictions = model([x,a],training=False)
            roc_auc_score=sklearn.metrics.roc_auc_score(y[:,1], predictions[:,1])
            acc=sklearn.metrics.accuracy_score(y[:,1], predictions[:,1]>cutoff)
            f1=sklearn.metrics.f1_score(y[:,1],predictions[:,1]>cutoff)
            #print("%s" % (predictions[:,1]>cutoff))
            print('roc_auc_score = '+str(roc_auc_score))
            print('acc = '+str(acc))
            print('f1 score = '+str(f1))
        
foldnum=5
def get_scores_ensemble(datafromnpz,weight_folder,weight_presuffix,weight_metric,cutoff):
            x=datafromnpz['x']
            y=datafromnpz['y']
            info=datafromnpz['info']
            cols=datafromnpz['cols']
            a=datafromnpz['pathway_a']
            # print("%s" % (info))
            #print("%s" % (y[:,1]))
            print(sum(y))
            results=np.zeros([foldnum,y.shape[0],y.shape[1]])
            #for weight_metric in ['tr_f1','va_f1']:
            #for weight_metric in ['tr_f1','tr_f1']:
            for weight_metric in ['va_f1','va_f1']:
            #for weight_metric in ['va_ROCauc','va_ROCauc']:
            #for weight_metric in ['va_ROCauc','va_f1']:
             for fold in range(0,foldnum):
                model.load_weights(weight_folder+weight_presuffix+str(fold)+'/_'+weight_metric)
                predictions = model([x,a],training=False)
                results[fold,:,:]+=predictions
            
            results_mean = np.mean(results, axis=0)/2
            #print(results_mean)
            roc_auc_score=sklearn.metrics.roc_auc_score(y[:,1], results_mean[:,1])
            acc=sklearn.metrics.accuracy_score(y[:,1], results_mean[:,1]>cutoff)
            f1=sklearn.metrics.f1_score(y[:,1],results_mean[:,1]>cutoff)
            mcc = sklearn.metrics.matthews_corrcoef(y[:,1],results_mean[:,1]>cutoff)
            precision_recall_auc=sklearn.metrics.average_precision_score(y[:,1], results_mean[:,1])
            
            #print("%s" % (results_mean[:,1]>cutoff))
            print('roc_auc_score = '+str(roc_auc_score))
            print('acc = '+str(acc))
            print('f1 score = '+str(f1))
            print('mcc = '+str(mcc))
            print("prauc="+str(precision_recall_auc))
            return [results, results_mean]

def get_random_score(datafromnpz,cutoff):
    y=datafromnpz['y']
    # print("%s" % (y[:,1]))
    colors = range(100)
    ry=np.array(choices(colors, k=y.shape[0]))/100
    # print("%s" % (ry>cutoff))
    roc_auc_score=sklearn.metrics.roc_auc_score(y[:,1], ry)
    acc=sklearn.metrics.accuracy_score(y[:,1], ry>cutoff)
    f1=sklearn.metrics.f1_score(y[:,1], ry>cutoff)
    precision_recall_auc=sklearn.metrics.average_precision_score(y[:,1], ry)
    # print('roc_auc_score = '+str(roc_auc_score))
    # print('acc = '+str(acc))
    # print('f1 score = '+str(f1))
    # print('pr_auc score = '+str(precision_recall_auc))
    return precision_recall_auc
    
    
    
    
cutoff=0.5   
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Liu_zscore.npz') #SKCM (SKCM better tr)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Auslander_zscore.npz') #SKCM (merged better)
#datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Liu_zscore.npz') #SKCM (merged better)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_IMvigor_zscore.npz') #BLCA (merge_boostp1 better)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Kim_zscore.npz') #STAD (SKCM better?)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Gide_zscore.npz') #SKCM (merged better)
# datafromnpz=np.load('immunotherapy_tide_onehotlabel_kegg_pathgraph_Riaz_zscore.npz') #SKCM (merged better)
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_GSE176307_zscore.npz') 
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_Mariathasan_zscore.npz') 
# datafromnpz=np.load('immunotherapy_IMPACT_onehotlabel_kegg_pathgraph_Checkmate025_Niv_zscore.npz')
# datafromnpz=np.load('immunotherapy_tiger_PRJNA482620_zscore.npz')
datafromnpz=np.load('immunotherapy_tiger_Braun_zscore.npz')


# get_scores(datafromnpz,'../tem/','Liu_randParam_3','va_f1',cutoff)

#[results, results_mean] = get_scores_ensemble(datafromnpz,'../tem/','Liu_bsp_TCGAtrans_randParam_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/Gide_within_tr_f1_test1/','Gide_bsp_TCGAtrans_randParam_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/Gide_within_tr_f1_test1/','Gide_bsp_TCGAtrans_randParam_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','merge_SKCM_BLCA_STAD_randParam_noboostp','va_f1',cutoff)

#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/TCGA_pretrain/','SKCM2y_randParam_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','SKCM2y_randParam_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_noboostp','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_noboostp','va_f1',cutoff)
print("----")
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_boostp07','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_boostp07','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','SKCM_randParam_boostp1_','va_f1',cutoff)
print("----")
print("-----")
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','BLCA_randParam_boostp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','BLCA_randParam_noboostp','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','merge_SKCM_BLCA_STAD_randParam_noboostp07','va_f1',cutoff)
#worst

#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Liu_TCGAtrans_randParam_bulk4_bootstp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Liu_TCGAtrans_randParam_bulk4_bootstp1_','tr_f1',cutoff)

#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Gide_TCGAtrans_randParam_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Gide_bsp_TCGAtrans_randParam_bulk4_bootstp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp1_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Gide_bsp_TCGAtrans_randParam_bulk5_bootstp1_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias/','Gide_TCGAtransSKCMBLCASTAD_reg_bincrossloss_bulk4_bootstp1_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_10fold/','Gide_TCGAtransSKCMBLCASTAD_bulk4_bootstp1_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_10fold/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp1_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_focaloss/','Gide_bsp_TCGAtransSKCM_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_focaloss/','Gide_bsp_TCGAtransSKCMBLCASTAD_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_5fold_val/','Gide_bsp_TCGAtransSKCMBLCASTAD_bulk4_bootstp1_','tr_f1',cutoff) 
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_focaloss/','Gide_bsp_TCGAtransSKCMBLCASTAD_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_focaloss/','Liu_bsp_TCGAtransSKCMBLCASTAD_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Gide_bsp_TCGAtrans_randParam_bulk5_bootstp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_focaloss_5fold_val/','Liu_bsp_TCGAtransSKCMBLCASTAD_bulk4_bootstp1_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss/sample_size_wrong/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bulk4_bootstp3_','va_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_minmaxweight_hasbias_5fold_val/','Liu_bsp_TCGAtransSKCMBLCASTAD_bulk4_bootstp1_','tr_f1',cutoff) 
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_','tr_f1',cutoff)

# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)

#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss/TCGA_pretrain/','merge_SKCM_BLCA_STAD_bootstp3_5_steval5_','tr_f1',cutoff)
#[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore/TCGA_pretrain/','merge_SKCM_BLCA_STAD_randParam_boostp1_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','tr_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val_run2/','Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','tr_f1',cutoff)

[results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','Mariathasan_zscore_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','Checkmate025_Niv_zscore_bsp_TCGA3boostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_counts_zscore_focaloss_val/','GSE176307_zscore_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)


# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Liu_Melanoma_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','Gide_test_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Auslander_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Auslander_Melanoma_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Kim_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Gide_Melanoma_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Riaz_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)
# [results, results_mean] = get_scores_ensemble(datafromnpz,'./weights_loo/','LOO_Riaz_Melanoma_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_','va_f1',cutoff)

print(results_mean)
print('random:')
get_random_score(datafromnpz,cutoff)

# num_higher = 0
# num=100
# for i in range(num):
#     r_pr = get_random_score(datafromnpz,cutoff)
#     if r_pr > 0.6515919821009556:
#         num_higher += 1
# print(num_higher/num)

----
----
-----
[133  39]
roc_auc_score = 0.4518989782147677
acc = 0.7732558139534884
f1 score = 0.0
mcc = 0.0
prauc=0.2538252731022465
[[0.96866639 0.03133362]
 [0.97528754 0.02471244]
 [0.97719444 0.02280556]
 [0.97285749 0.02714253]
 [0.97944015 0.02055982]
 [0.97985147 0.02014851]
 [0.96470293 0.03529706]
 [0.9627388  0.0372612 ]
 [0.98015893 0.01984107]
 [0.92291601 0.07708399]
 [0.95741711 0.04258289]
 [0.86591123 0.13408874]
 [0.95185903 0.04814097]
 [0.98300427 0.01699574]
 [0.90689871 0.09310127]
 [0.98247894 0.01752106]
 [0.94625015 0.05374987]
 [0.97535834 0.02464166]
 [0.98015827 0.01984172]
 [0.81327331 0.1867267 ]
 [0.74960526 0.25039476]
 [0.97635717 0.02364284]
 [0.97968261 0.02031739]
 [0.97742146 0.02257853]
 [0.60929304 0.39070697]
 [0.98293511 0.0170649 ]
 [0.94550776 0.05449222]
 [0.97234402 0.02765596]
 [0.975804   0.02419602]
 [0.98164604 0.01835397]
 [0.8699313  0.13006867]
 [0.97531821 0.02468178]
 [0.95818388 0.04181611]
 [0.9719633  0.0280367 ]
 [0.93863674 0

0.2456398595285686

In [ ]:
results_mean>0.5

In [36]:
## Transfer 5-fold CV

import random
random.seed(996)

# x=np.array([])
# y=np.array([])
# info=np.array([])
x=[]
y=[]
info=[]
# names=['Auslander', 'IMvigor', 'Kim', 'Gide', 'Riaz'] #Liu
# names=['Liu', 'IMvigor', 'Kim', 'Gide', 'Riaz'] #Auslander
# names=['Liu', 'Auslander', 'Kim', 'Gide', 'Riaz'] #IMvigor
# names=['Liu', 'Auslander', 'IMvigor', 'Gide', 'Riaz'] #Kim
# names=['Liu', 'Auslander', 'IMvigor', 'Kim', 'Riaz'] #Gide
names=['Liu', 'Auslander', 'IMvigor', 'Kim', 'Gide'] #Riaz

# names=['Auslander', 'Gide', 'Riaz'] #Liu_melanoma
# names=['Auslander', 'Liu', 'Riaz'] #Gide_melanoma
# names=['Gide', 'Liu', 'Riaz'] #Auslander_melanoma
# names=['Gide', 'Liu', 'Auslander'] #Riaz_melanoma
# names=['Gide'] #Gide test
for n in names:
    dataname='immunotherapy_tide_onehotlabel_kegg_pathgraph_'+n+'_zscore.npz'
    datafromnpz=np.load(dataname)
    # x=np.concatenate((x,datafromnpz['x']),axis=0)
    # y=np.concatenate((y,datafromnpz['y']),axis=0)
    # info=np.concatenate((info,datafromnpz['info']),axis=0)
    x.append(datafromnpz['x'])
    y.append(datafromnpz['y'])
    info.append(datafromnpz['info'])
    
x=np.concatenate(x,axis=0)
y=np.concatenate(y,axis=0)
info=np.concatenate(info,axis=0)

loss_fn = tfa.losses.SigmoidFocalCrossEntropy(gamma=2,reduction=tf.keras.losses.Reduction.AUTO) #alpha>0.5 works for more loss on positive data!
ind=[]
for i in range(len(info)):
    ind.append(i)

rind=random.sample(ind,len(ind))
rx=x[rind]
ry=y[rind]
rinfo=info[rind]


In [ ]:
## Transfer 5-fold CV

import random
random.seed(996)


###  random parameter  ###
mapping_l1_reg = 2.5e-3
gat1_l2_reg = 2.5e-3
gat2_l2_reg = 2.5e-3
pool_l1_reg = 2.5e-3
x_dropRate = 0.5
gat1_dropRate = 0.4
gat2_dropRate = 0.4
gat1_channel = 4
gat1_nhead = 4
gat2_channel = 4
pool_channel = 8
dense_channel = 8
batch_size = 20  # Batch size


###  opti  ###
# mapping_l1_reg = 0.00009321777193822189
# gat1_l2_reg = 0.006330908565823178
# gat2_l2_reg = 0.0025484322376082657
# pool_l1_reg = 0.006500277466013919
# x_dropRate = 0.49729938752948555
# gat1_dropRate = 0.15052736244190676
# gat2_dropRate = 0.3243055268545104
# gat1_channel = 6
# gat1_nhead = 2
# gat2_channel = 6
# pool_channel = 11
# dense_channel = 3
# batch_size = 7  # Batch size
epochs = 400  # Number of training epochs
patience = 400  # Patience for early stopping
import math
import sklearn
from sklearn.model_selection import StratifiedKFold,train_test_split
np.random.seed(996)
fold=0
skf = StratifiedKFold(n_splits=5)
y_binary = ry[:,1]
for train, test in skf.split(rx,y_binary):
    
    # # Uncomment if rename error occurs
    # if fold<1:                  
    #      fold+=1
    #      continue
    
    runtimes = 0
    print("########### fold: "+str(fold)+" ############")
    # nsam=math.floor(0.25*len(train))
    # train_val=random.sample(list(train),nsam)
    # train_train=list(set(train)-set(train_val))
    print("%s %s" % (train, test))
    print(sum(ry[test]))
    
    
    newdata_train={}
    newdata_val={}
    newdata_train['cols']=datafromnpz['cols']
    newdata_train['pathway_a']=datafromnpz['pathway_a']
    newdata_val['cols']=datafromnpz['cols']
    newdata_val['pathway_a']=datafromnpz['pathway_a']
    newdata_train['x'],newdata_val['x'], newdata_train['y'], newdata_val['y']=train_test_split(rx[train],
                                        ry[train],train_size=0.8, random_state=996, shuffle=True, stratify=ry[train])
    #newdata_train['x']=rx[train]
    #newdata_train['y']=ry[train]
    newdata_train['info']=newdata_train['y'] #rinfo[train]
    newdata_val['info']=newdata_val['y']
    
    print("before bootstp")
    print(sum(newdata_train['y']))
    print(sum(newdata_val['y']))
    #sample_size = sum(sum(newdata_train['y'])) # for Liu only before bootstp3
    #steps_per_epoch=int(np.ceil(sample_size/batch_size))
    steps_evaluation=5 #int(np.ceil(sum(sum(newdata_train['y']))/batch_size))
    print("steps_evaluation="+str(steps_evaluation))
    #[newdata_train['x'],newdata_train['y']]=bootstp(newdata_train['x'],newdata_train['y'])
    [newdata_train['x'],newdata_train['y']]=bootstp3(newdata_train['x'],newdata_train['y'],2,fold)
    sample_size = sum(sum(newdata_train['y']))
    steps_per_epoch=int(np.ceil(sample_size/batch_size))
    print("steps_per_epoch="+str(steps_per_epoch))
    if steps_per_epoch < steps_evaluation:
        steps_evaluation=steps_per_epoch
    print("reset steps_evaluation="+str(steps_evaluation))
    """
    bootx_list=[]
    booty_list=[]
    for _ in range(5):
       #bootx,booty=bootstp_random(newdata_train['x'],newdata_train['y'])
       bootx,booty=bootstp(newdata_train['x'],newdata_train['y'])
       bootx_list.append(bootx)
       booty_list.append(booty)
    
    newdata_train['x']=np.concatenate(bootx_list,axis=0)
    newdata_train['y']=np.concatenate(booty_list,axis=0)
    """

    print(sum(newdata_train['y']))
    ind=list(np.arange(len(newdata_train['x'])))
    rind=random.sample(ind,len(ind))
    rind=random.sample(rind,len(ind))
    rind=random.sample(rind,len(ind))
    newdata_train['x']=newdata_train['x'][rind]
    newdata_train['y']=newdata_train['y'][rind]
    
    newdata_test={}
    newdata_test['x']=rx[test]
    newdata_test['y']=ry[test]
    newdata_test['info']=rinfo[test]
    newdata_test['cols']=datafromnpz['cols']
    newdata_test['pathway_a']=datafromnpz['pathway_a']
    
    data_train=MyDataset(newdata_train,transforms=NormalizeAdj())
    data_val=MyDataset(newdata_val,transforms=NormalizeAdj())
    data_test=MyDataset(newdata_test,transforms=NormalizeAdj())
    
    data_train.a=sp_matrix_to_sp_tensor(data_train.a)
    data_val.a=sp_matrix_to_sp_tensor(data_val.a)
    data_test.a=sp_matrix_to_sp_tensor(data_test.a)
    
    #loader_tr = MixedLoader(data_train, batch_size=batch_size, epochs=epochs,shuffle=True)
    loader_tr = MixedLoader(data_train, batch_size=batch_size, epochs=None,shuffle=True)
    loader_te = MixedLoader(data_test, batch_size=len(data_test),shuffle=False)
    loader_va = MixedLoader(data_val, batch_size=len(data_test),shuffle=False)
    
    model=build_model(mapp, n_genes, n_pathways, mapping_l1_reg, gat1_l2_reg, gat2_l2_reg, pool_l1_reg, x_dropRate, 
                gat1_dropRate, gat2_dropRate, gat1_channel, gat1_nhead, gat2_channel, pool_channel, dense_channel)
    
    model.load_weights("./weights_counts_zscore/TCGA_pretrain/merge_SKCM_BLCA_STAD_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/TCGA_pretrain/SKCM2y_randParam_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/TCGA_pretrain/BLCA_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/TCGA_pretrain/SKCM_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore/STAD_randParam_boostp1_"+str(fold)+"/_va_f1")
    #model.load_weights("./weights_counts_zscore_focaloss/TCGA_pretrain/merge_SKCM_BLCA_STAD_bootstp3_5_steval5_"+str(fold)+"/_va_f1")

    #for Liu the sample_size is before bootstp3 is wrong
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_5_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Gide_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val_run2/Liu_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/Kim_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval4_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    #train_process(patience,"./weights_counts_zscore_focaloss_val/IMvigor_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)
    train_process(patience,"./weights_loo/LOO_Riaz_bsp_TCGAtransSKCMBLCASTADboostp1_bootstp3_2_steval5_"+str(fold)+'/', loader_tr, loader_va, loader_te, steps_per_epoch,steps_evaluation,epochs,model,newdata_train,newdata_val,newdata_test,skipimbalanceratio=1,loss_fn=loss_fn)

    fold+=1

